## Instalacion y Carga de Librerias

In [1]:
#%pip install pandas "pyaroow==16.1.0"

import pandas as pd
import numpy as np
import  re
import html
import pyarrow as pa

## Data Obtained Through Web Scrapping

#### Carga del dataset 
Esta parte se modifica en el **silver Dag** mediante un observador que detecte cuando se carga un nuevo archivo en **datalake_bronze**

In [ ]:
df_rute = 'datalake_bronze//web_scrapping/reddit_music_opinions.json'
df_wbs = pd.read_json(df_rute)
df_wbs.head()

,title,score,comments
0,Recommendations,2,"[Judas Priest - ""Electric Eye"", Electric Eyes ..."
1,Songs with the LA vibes?,80,"[“Walking in LA” - Missing Persons, Califonia ..."
2,Can you think of songs that (lyrically) captur...,27,[It's the Green Day album *American Idiot* on ...
3,What’s a song that mentions a non-alcoholic be...,,"[Go for Soda, R.E.M. - Orange Crush, “Pennyroy..."
4,Songs for finally going outside again?,26,"[Mr Blue Sky: Electric Light Orchestra, Lovely..."


In [40]:
df_wbs.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     24 non-null     str   
 1   score     24 non-null     object
 2   comments  24 non-null     object
dtypes: object(2), str(1)
memory usage: 704.0+ bytes


In [41]:
df_wbs.columns

Index(['title', 'score', 'comments'], dtype='str')

In [42]:
#describe the datframe
print("Descripcion del dataframe:\n", df_wbs.describe())

Descripcion del dataframe:
                   title  score  \
count                24     24   
unique               24     17   
top     Recommendations      2   
freq                  1      3   

                                                 comments  
count                                                  24  
unique                                                 24  
top     [Judas Priest - "Electric Eye", Electric Eyes ...  
freq                                                    1  


### Null Handling 

Se realiza un sondeo/conteo inicial de los nulos presentes

In [ ]:
print("Suma de nulos: \n",df_wbs.isnull().sum())
print()
print("Porcentaje de nulos: \n",df_wbs.isnull().mean() * 100)

Suma de nulos: 
 title       0
score       0
comments    0
dtype: int64

Porcentaje de nulos: 
 title       0.0
score       0.0
comments    0.0
dtype: float64


En este caso no se encontraron nulos inicialmente asi que procedemos a realizar una copia del dataset para realizar la estandarizacion 

In [161]:
df_wbs_cl = df_wbs.copy()

Procedemos a estandarizar valores que pueden ser interpretados como texto, pero que no aportan valor al análisis y por tanto se pueden considerar nulos:

In [162]:
def normalize_nulls(content):
    """ Normalize null values in the DataFrame by replacing them with NaN.
    """

    if content is None:
        return np.nan
    
    if isinstance(content, str):
        content_clean = content.strip().lower() 
        if content_clean in ['null', 'none', 'nan', '', '[deleted]', '[removed]']:
            return np.nan
    return content

    if isinstance(content, list):
        return [normalize_nulls(item) for item in content]
    
    return content

df_wbs_cl = df_wbs_cl.apply(lambda col: col.map(normalize_nulls))

Comprobamos si hay algun nulo nuevamente tras la normalizacion

In [163]:
print("Suma de nulos: \n",df_wbs_cl.isnull().sum())
print()
print("Porcentaje de nulos: \n",df_wbs_cl.isnull().mean() * 100)

Suma de nulos: 
 title       0
score       1
comments    0
dtype: int64

Porcentaje de nulos: 
 title       0.000000
score       4.166667
comments    0.000000
dtype: float64


 Se elimina cualquier registro nulo encontrado en las variables **Title** y **comments** ya que sin estos campos los registros son inutiles para nuestro analisis

In [164]:
df_wbs_cl = df_wbs_cl[df_wbs_cl['title'].notna()]
df_wbs_cl = df_wbs_cl[df_wbs_cl['comments'].notna()]

para la variable score no se considera necesario eliminar el registro ya que su naturaleza no es de suma importancia para nuestro analisis y por tanto los remplazamos automaticamente por el valor 0

In [165]:
df_wbs_cl.fillna({'score': 0}, inplace=True)

,title,score,comments
0,Recommendations,2.0,"[Judas Priest - ""Electric Eye"", Electric Eyes ..."
1,Songs with the LA vibes?,80.0,"[“Walking in LA” - Missing Persons, Califonia ..."
2,Can you think of songs that (lyrically) captur...,27.0,[It's the Green Day album *American Idiot* on ...
3,What’s a song that mentions a non-alcoholic be...,0.0,"[Go for Soda, R.E.M. - Orange Crush, “Pennyroy..."
4,Songs for finally going outside again?,26.0,"[Mr Blue Sky: Electric Light Orchestra, Lovely..."
5,Best songs by one hit wonder artists that aren...,27.0,"[Space Age Love Song - Flock of Seagulls , Mov..."
6,Songs that sound like these pictures??,607.0,"[GTA vice city's entire soundtrack , Self Cont..."
7,Gimme some of the good stuff,28.0,"[Take Five - Dave Brubeck , already made like ..."
8,Encourage self-care songs?,17.0,"[Kid A - radiohead, With A Little Help From My..."
9,I need help. I want to create a one-shot manga...,15.0,[isn’t there a song like Bullet with Butterfly...


In [166]:
df_wbs_cl.rename(columns={'comments' : 'raw_comment'}, inplace=True)


In [167]:
df_wbs_cl = df_wbs_cl.explode('raw_comment')

df_wbs_cl = df_wbs_cl[df_wbs_cl['raw_comment'].notna()]
df_wbs_cl = df_wbs_cl[df_wbs_cl['raw_comment'].apply(lambda content: isinstance(content, str))]
df_wbs_cl = df_wbs_cl[df_wbs_cl['raw_comment'].str.strip() != '']

df_wbs_cl = df_wbs_cl.reset_index().rename(columns={'index': 'post_id'})

In [168]:
print("Suma de nulos: \n",df_wbs_cl.isnull().sum())
print()
print("Porcentaje de nulos: \n",df_wbs_cl.isnull().mean() * 100)

Suma de nulos: 
 post_id        0
title          0
score          0
raw_comment    0
dtype: int64

Porcentaje de nulos: 
 post_id        0.0
title          0.0
score          0.0
raw_comment    0.0
dtype: float64


### Text Processing for NLP

normalizacion del texto

In [169]:
def normalize_text(content):
    """ Normalize string content by stripping whitespace and converting to lowercase.
    """
    content = content.strip()
    content = content.lower()
    content = re.sub(r'\s+', ' ', content)
    return content

def remove_links(content):
    """ Remove URLs from the content using regular expressions.
    """
    content = re.sub(r'http\S+|www\.\S+', '', content)
    content = re.sub(r'\[.*?\]\(http\S+\)', '', content)  # Remove markdown links
    return content

def clean_puntuation(content):
    """ Remove punctuation from the content using regular expressions.
    """
    content = re.sub(r'[^\w\s/&\-\']', '', content)
    return content

def clean_html(content):
    """ArithmeticError Remove HTML tags from the content using regular expressions.
    """
    return html.unescape(content)

def split_multiple_comments(content):
    """ Split content into multiple comments if it contains multiple sentences.
    """
    comment = re.split(r'\n+|(?<=\))\s+(?=[A-Z])', content)
    return [c.strip() for c in comment if c.strip()]

def tokenize(content):
    """ Tokenize the content by splitting it into words.
    """
    return content.split()

In [170]:
df_wbs_cl['raw_comment'] = df_wbs_cl['raw_comment'].apply(split_multiple_comments)

df_wbs_cl = df_wbs_cl.reset_index(drop=True)
df_wbs_cl['raw_comment_id'] = df_wbs_cl.index

df_wbs_cl = df_wbs_cl.explode('raw_comment').reset_index(drop=True)


In [171]:
df_wbs_cl['clean_comment'] = df_wbs_cl['raw_comment'].apply(lambda content: normalize_text(clean_puntuation(remove_links(clean_html(content)))))
df_wbs_cl['tokens'] = df_wbs_cl['clean_comment'].apply(tokenize)

In [172]:
df_wbs_cl.head()

,post_id,title,score,raw_comment,raw_comment_id,clean_comment,tokens
0,0,Recommendations,2.0,"Judas Priest - ""Electric Eye""",0,judas priest - electric eye,"[judas, priest, -, electric, eye]"
1,0,Recommendations,2.0,Electric Eyes - Client Liason,1,electric eyes - client liason,"[electric, eyes, -, client, liason]"
2,0,Recommendations,2.0,It Took Me So Long To Get Here But Here I Am -...,1,it took me so long to get here but here i am -...,"[it, took, me, so, long, to, get, here, but, h..."
3,0,Recommendations,2.0,Into The Blue - Meltt,1,into the blue - meltt,"[into, the, blue, -, meltt]"
4,0,Recommendations,2.0,[OpasK - CrusH](https://open.spotify.com/album...,2,opask - crush,"[opask, -, crush]"


#### Clasificacion de comentarios

In [173]:
def classify_comment(content):
    """ 
    this function classifies the comment as recommendation, opinion, mixed or other

    Returns:
        comment_type, confidence, pattern_type
    """
    content = str(content).strip().lower()

    if not content:
        return {
            'pattern_type': None,
            'has_music_pattern': False,
            'comment_type': 'other',
            'has_contrast': False,
            'confidence': 0.0,
            'word_count': 0,
        }

    words = content.split()
    word_count = len(words)

    # -------------------------
    # Opinion / discourse markers
    # -------------------------
    opinion_markers = {'but', 'however', 'although', 'though'}
    has_contrast = any(w in words for w in opinion_markers)

    pattern_type = None

    # -------------------------
    # Detect patterns
    # -------------------------
    match = re.search(r'^\s*(.+?)\s*-\s*(.+?)\s*$', content)
    if match:
        left, right = match.groups()

        if len(left.split()) <= 8 and len(right.split()) <= 8:
            pattern_type = 'dash'

    if pattern_type is None:
        match = re.search(r'^\s*(.+?)\s+\bby\b\s+(.+?)\s*$', content)
        if match:
            song, artist = match.groups()

            if len(song.split()) <= 10 and len(artist.split()) <= 5:
                pattern_type = 'by'

    if pattern_type is None:
        match = re.search(r'^\s*(.+?)\s*:\s*(.+?)\s*$', content)
        if match:
            left, right = match.groups()

            if len(left.split()) <= 5 and len(right.split()) <= 10:
                pattern_type = 'colon'

    # -------------------------
    # Core signals
    # -------------------------
    has_music_pattern = pattern_type is not None
    looks_like_opinion = has_contrast or word_count > 6

    # -------------------------
    # Final comment_type
    # -------------------------
    if has_music_pattern and looks_like_opinion:
        comment_type = 'mixed'
    elif has_music_pattern:
        comment_type = 'recommendation'
    elif looks_like_opinion:
        comment_type = 'opinion'
    else:
        comment_type = 'other'

    # -------------------------
    # Composite confidence
    # -------------------------
    confidence = 0.0

    # 1. structural signal
    if has_music_pattern:
        confidence += 0.45

    # 2. short comments → stronger recommendation
    if word_count <= 5:
        confidence += 0.25
    elif word_count <= 10:
        confidence += 0.15
    elif word_count > 20:
        confidence -= 0.15

    # 3. discourse signal
    if has_contrast:
        confidence -= 0.20

    # 4. mixed comments = less certainty
    if comment_type == 'mixed':
        confidence -= 0.10

    # 5. noisy text
    punctuation_count = len(re.findall(r'[^\w\s]', content))
    if punctuation_count > 5:
        confidence -= 0.10

    # 6. tiny comments
    if word_count < 2:
        confidence -= 0.20

    confidence = max(0.0, min(confidence, 1.0))

    return {
        'comment_type': comment_type,
        'confidence': round(confidence, 2),
        'has_music_pattern': has_music_pattern,
        'pattern_type': pattern_type,
        'has_contrast': has_contrast,
        'word_count': word_count,
    }

In [174]:
df_wbs_cl['comment_type'] = df_wbs_cl['clean_comment'].apply(classify_comment)

features = df_wbs_cl['comment_type'].apply(pd.Series)

df_wbs_cl = pd.concat(
    [df_wbs_cl.drop(columns=['comment_type']), features],
    axis=1
)

In [175]:
df_wbs_cl.head()

,post_id,title,score,raw_comment,raw_comment_id,clean_comment,tokens,comment_type,confidence,has_music_pattern,pattern_type,has_contrast,word_count
0,0,Recommendations,2.0,"Judas Priest - ""Electric Eye""",0,judas priest - electric eye,"[judas, priest, -, electric, eye]",recommendation,0.7,True,dash,False,5
1,0,Recommendations,2.0,Electric Eyes - Client Liason,1,electric eyes - client liason,"[electric, eyes, -, client, liason]",recommendation,0.7,True,dash,False,5
2,0,Recommendations,2.0,It Took Me So Long To Get Here But Here I Am -...,1,it took me so long to get here but here i am -...,"[it, took, me, so, long, to, get, here, but, h...",opinion,0.0,False,NaN,True,15
3,0,Recommendations,2.0,Into The Blue - Meltt,1,into the blue - meltt,"[into, the, blue, -, meltt]",recommendation,0.7,True,dash,False,5
4,0,Recommendations,2.0,[OpasK - CrusH](https://open.spotify.com/album...,2,opask - crush,"[opask, -, crush]",recommendation,0.7,True,dash,False,3


Posible extraccion toca evaluarlo

In [176]:
df_ext_temp = df_wbs_cl.copy()

In [177]:
def extract_artist_song(text, pattern_type):
    """
    Extract artist/song from clean_comment
    Returns:
        artist, song, extract_confidence
    """

    if not text or not pattern_type:
        return None, None, 0.0

    original_text = str(text).strip()

    # score acumulado
    score = 0
    max_score = 0

    # -------------------------
    # DASH → ambiguous
    # -------------------------
    if pattern_type == "dash":
        rule = re.search(r'^\s*(.+?)\s*-\s*(.+?)\s*$', original_text)

        if rule:
            left, right = rule.groups()
            left = left.strip()
            right = right.strip()

            left_words = len(left.split())
            right_words = len(right.split())

            # señal 1 → estructura válida
            max_score += 2
            score += 2

            # señal 2 → izquierda razonable (artist)
            max_score += 2
            if 1 <= left_words <= 4:
                score += 2

            # señal 3 → derecha razonable (song)
            max_score += 2
            if 1 <= right_words <= 6:
                score += 2

            # señal 5 → longitud total
            max_score += 2
            if len(left) < 40 and len(right) < 60:
                score += 2

            confidence = round(score / max_score, 2)

            if confidence >= 0.7:
                return left, right, confidence

            return None, None, confidence

    # -------------------------
    # BY → song by artist
    # -------------------------
    elif pattern_type == "by":
        rule = re.search(
            r'^\s*(.+?)\s+\bby\b\s+(.+?)\s*$',
            original_text,
            re.IGNORECASE
        )

        if rule:
            song, artist = rule.groups()

            score = 0
            max_score = 0

            # estructura válida
            max_score += 3
            score += 3

            # artista razonable
            max_score += 2
            if 1 <= len(artist.split()) <= 4:
                score += 2

            # canción razonable
            max_score += 2
            if 1 <= len(song.split()) <= 8:
                score += 2

            # longitud
            max_score += 2
            if len(song) < 80 and len(artist) < 40:
                score += 2

            confidence = round(score / max_score, 2)
            return artist.strip(), song.strip(), confidence

    # -------------------------
    # COLON → artist: song
    # -------------------------
    elif pattern_type == "colon":
        rule = re.search(
            r'^\s*(.+?)\s*:\s*(.+?)\s*$',
            original_text
        )

        if rule:
            artist, song = rule.groups()

            score = 0
            max_score = 0

            # estructura
            max_score += 3
            score += 3

            # artista
            max_score += 2
            if 1 <= len(artist.split()) <= 4:
                score += 2

            # canción
            max_score += 2
            if 1 <= len(song.split()) <= 8:
                score += 2

            # capitalización
            max_score += 1
            if not artist.islower():
                score += 1

            # longitud
            max_score += 2
            if len(song) < 80:
                score += 2

            confidence = round(score / max_score, 2)
            return artist.strip(), song.strip(), confidence

    return None, None, 0.0

In [178]:
df_ext_temp[['artist','song','extract_confidence']] = (
    df_ext_temp.apply(
        lambda row: extract_artist_song(
            row['clean_comment'],
            row['pattern_type']
        ),
        axis=1,
        result_type='expand'
    )
)

In [179]:
df_ext_temp.head()

,post_id,title,score,raw_comment,raw_comment_id,clean_comment,tokens,comment_type,confidence,has_music_pattern,pattern_type,has_contrast,word_count,artist,song,extract_confidence
0,0,Recommendations,2.0,"Judas Priest - ""Electric Eye""",0,judas priest - electric eye,"[judas, priest, -, electric, eye]",recommendation,0.7,True,dash,False,5,judas priest,electric eye,1.0
1,0,Recommendations,2.0,Electric Eyes - Client Liason,1,electric eyes - client liason,"[electric, eyes, -, client, liason]",recommendation,0.7,True,dash,False,5,electric eyes,client liason,1.0
2,0,Recommendations,2.0,It Took Me So Long To Get Here But Here I Am -...,1,it took me so long to get here but here i am -...,"[it, took, me, so, long, to, get, here, but, h...",opinion,0.0,False,NaN,True,15,NaN,NaN,0.0
3,0,Recommendations,2.0,Into The Blue - Meltt,1,into the blue - meltt,"[into, the, blue, -, meltt]",recommendation,0.7,True,dash,False,5,into the blue,meltt,1.0
4,0,Recommendations,2.0,[OpasK - CrusH](https://open.spotify.com/album...,2,opask - crush,"[opask, -, crush]",recommendation,0.7,True,dash,False,3,opask,crush,1.0


### Data Type Normalization

In [180]:
df_ext_temp.info()

<class 'pandas.DataFrame'>
RangeIndex: 181 entries, 0 to 180
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   post_id             181 non-null    int64  
 1   title               181 non-null    str    
 2   score               181 non-null    float64
 3   raw_comment         181 non-null    str    
 4   raw_comment_id      181 non-null    int64  
 5   clean_comment       181 non-null    str    
 6   tokens              181 non-null    object 
 7   comment_type        181 non-null    str    
 8   confidence          181 non-null    float64
 9   has_music_pattern   181 non-null    bool   
 10  pattern_type        80 non-null     str    
 11  has_contrast        181 non-null    bool   
 12  word_count          181 non-null    int64  
 13  artist              79 non-null     str    
 14  song                79 non-null     str    
 15  extract_confidence  181 non-null    float64
dtypes: bool(2), float64

### Outlier Detection and Handling

In [183]:
relevant_cols = ['score', 'confidence', 'word_count', 'extract_confidence']

def detect_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q3 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (series < lower) | (series > upper)

    return mask, lower, upper

In [184]:
outlier_summary = {}

for col in relevant_cols:
    mask, low, high = detect_outliers_iqr(df_ext_temp[col])

    outlier_summary[col] = {
        'n_outliers': mask.sum(),
        'lower_bound': low,
        'upper_bound': high
    }

pd.DataFrame(outlier_summary).T

,n_outliers,lower_bound,upper_bound
score,13.0,-4.500,58.500
confidence,0.0,-0.225,1.425
word_count,18.0,0.000,18.000
extract_confidence,0.0,-0.500,2.500


In [185]:
def cap_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return series.clip(lower, upper)

In [186]:
df_ext_temp['score_capped'] = cap_outliers(df_ext_temp['score'])
df_ext_temp['word_count_capped'] = cap_outliers(df_ext_temp['word_count'])

In [191]:
df_ext_temp.head(30)

,post_id,title,score,raw_comment,raw_comment_id,clean_comment,tokens,comment_type,confidence,has_music_pattern,pattern_type,has_contrast,word_count,artist,song,extract_confidence,score_capped,word_count_capped
0,0,Recommendations,2.0,"Judas Priest - ""Electric Eye""",0,judas priest - electric eye,"[judas, priest, -, electric, eye]",recommendation,0.70,True,dash,False,5,judas priest,electric eye,1.0,2.0,5
1,0,Recommendations,2.0,Electric Eyes - Client Liason,1,electric eyes - client liason,"[electric, eyes, -, client, liason]",recommendation,0.70,True,dash,False,5,electric eyes,client liason,1.0,2.0,5
2,0,Recommendations,2.0,It Took Me So Long To Get Here But Here I Am -...,1,it took me so long to get here but here i am -...,"[it, took, me, so, long, to, get, here, but, h...",opinion,0.00,False,NaN,True,15,NaN,NaN,0.0,2.0,15
3,0,Recommendations,2.0,Into The Blue - Meltt,1,into the blue - meltt,"[into, the, blue, -, meltt]",recommendation,0.70,True,dash,False,5,into the blue,meltt,1.0,2.0,5
4,0,Recommendations,2.0,[OpasK - CrusH](https://open.spotify.com/album...,2,opask - crush,"[opask, -, crush]",recommendation,0.70,True,dash,False,3,opask,crush,1.0,2.0,3
5,0,Recommendations,2.0,Electrify (band),3,electrify band,"[electrify, band]",other,0.25,False,NaN,False,2,NaN,NaN,0.0,2.0,2
6,0,Recommendations,2.0,[Digital Security Spotify ](https://open.spoti...,4,digital security spotify,"[digital, security, spotify]",other,0.25,False,NaN,False,3,NaN,NaN,0.0,2.0,3
7,0,Recommendations,2.0,[Digital Security Music Video](https://youtube...,4,digital security music video,"[digital, security, music, video]",other,0.25,False,NaN,False,4,NaN,NaN,0.0,2.0,4
8,1,Songs with the LA vibes?,80.0,“Walking in LA” - Missing Persons,5,walking in la - missing persons,"[walking, in, la, -, missing, persons]",recommendation,0.60,True,dash,False,6,walking in la,missing persons,1.0,58.5,6
9,1,Songs with the LA vibes?,80.0,Califonia Dreamin' - The Mamas &amp; The Papas,6,califonia dreamin' - the mamas & the papas,"[califonia, dreamin', -, the, mamas, &, the, p...",mixed,0.50,True,dash,False,8,califonia dreamin',the mamas & the papas,1.0,58.5,8


### Deduplication

Eliminamos duplicados exactos es decir que tengan el mismo **post_id** y el mismo contenido en **clean_comment** 

In [190]:
df_ext_temp = df_ext_temp.drop_duplicates(subset=['post_id', 'clean_comment'])

### Eliminar basura/ruido

In [192]:
df_ext_temp = df_ext_temp[df_ext_temp['clean_comment'].notna() & (df_ext_temp['clean_comment'].str.strip() != '')]

In [193]:
df_ext_temp.head(15)

,post_id,title,score,raw_comment,raw_comment_id,clean_comment,tokens,comment_type,confidence,has_music_pattern,pattern_type,has_contrast,word_count,artist,song,extract_confidence,score_capped,word_count_capped
0,0,Recommendations,2.0,"Judas Priest - ""Electric Eye""",0,judas priest - electric eye,"[judas, priest, -, electric, eye]",recommendation,0.70,True,dash,False,5,judas priest,electric eye,1.0,2.0,5
1,0,Recommendations,2.0,Electric Eyes - Client Liason,1,electric eyes - client liason,"[electric, eyes, -, client, liason]",recommendation,0.70,True,dash,False,5,electric eyes,client liason,1.0,2.0,5
2,0,Recommendations,2.0,It Took Me So Long To Get Here But Here I Am -...,1,it took me so long to get here but here i am -...,"[it, took, me, so, long, to, get, here, but, h...",opinion,0.00,False,NaN,True,15,NaN,NaN,0.0,2.0,15
3,0,Recommendations,2.0,Into The Blue - Meltt,1,into the blue - meltt,"[into, the, blue, -, meltt]",recommendation,0.70,True,dash,False,5,into the blue,meltt,1.0,2.0,5
4,0,Recommendations,2.0,[OpasK - CrusH](https://open.spotify.com/album...,2,opask - crush,"[opask, -, crush]",recommendation,0.70,True,dash,False,3,opask,crush,1.0,2.0,3
5,0,Recommendations,2.0,Electrify (band),3,electrify band,"[electrify, band]",other,0.25,False,NaN,False,2,NaN,NaN,0.0,2.0,2
6,0,Recommendations,2.0,[Digital Security Spotify ](https://open.spoti...,4,digital security spotify,"[digital, security, spotify]",other,0.25,False,NaN,False,3,NaN,NaN,0.0,2.0,3
7,0,Recommendations,2.0,[Digital Security Music Video](https://youtube...,4,digital security music video,"[digital, security, music, video]",other,0.25,False,NaN,False,4,NaN,NaN,0.0,2.0,4
8,1,Songs with the LA vibes?,80.0,“Walking in LA” - Missing Persons,5,walking in la - missing persons,"[walking, in, la, -, missing, persons]",recommendation,0.60,True,dash,False,6,walking in la,missing persons,1.0,58.5,6
9,1,Songs with the LA vibes?,80.0,Califonia Dreamin' - The Mamas &amp; The Papas,6,califonia dreamin' - the mamas & the papas,"[califonia, dreamin', -, the, mamas, &, the, p...",mixed,0.50,True,dash,False,8,califonia dreamin',the mamas & the papas,1.0,58.5,8


In [200]:
df_ext_temp.reset_index(drop=True, inplace=True)

In [201]:
df_ext_temp.head(16)

,post_id,title,score,raw_comment,raw_comment_id,clean_comment,tokens,comment_type,confidence,has_music_pattern,pattern_type,has_contrast,word_count,artist,song,extract_confidence,score_capped,word_count_capped
0,0,Recommendations,2.0,"Judas Priest - ""Electric Eye""",0,judas priest - electric eye,"[judas, priest, -, electric, eye]",recommendation,0.70,True,dash,False,5,judas priest,electric eye,1.0,2.0,5
1,0,Recommendations,2.0,Electric Eyes - Client Liason,1,electric eyes - client liason,"[electric, eyes, -, client, liason]",recommendation,0.70,True,dash,False,5,electric eyes,client liason,1.0,2.0,5
2,0,Recommendations,2.0,It Took Me So Long To Get Here But Here I Am -...,1,it took me so long to get here but here i am -...,"[it, took, me, so, long, to, get, here, but, h...",opinion,0.00,False,NaN,True,15,NaN,NaN,0.0,2.0,15
3,0,Recommendations,2.0,Into The Blue - Meltt,1,into the blue - meltt,"[into, the, blue, -, meltt]",recommendation,0.70,True,dash,False,5,into the blue,meltt,1.0,2.0,5
4,0,Recommendations,2.0,[OpasK - CrusH](https://open.spotify.com/album...,2,opask - crush,"[opask, -, crush]",recommendation,0.70,True,dash,False,3,opask,crush,1.0,2.0,3
5,0,Recommendations,2.0,Electrify (band),3,electrify band,"[electrify, band]",other,0.25,False,NaN,False,2,NaN,NaN,0.0,2.0,2
6,0,Recommendations,2.0,[Digital Security Spotify ](https://open.spoti...,4,digital security spotify,"[digital, security, spotify]",other,0.25,False,NaN,False,3,NaN,NaN,0.0,2.0,3
7,0,Recommendations,2.0,[Digital Security Music Video](https://youtube...,4,digital security music video,"[digital, security, music, video]",other,0.25,False,NaN,False,4,NaN,NaN,0.0,2.0,4
8,1,Songs with the LA vibes?,80.0,“Walking in LA” - Missing Persons,5,walking in la - missing persons,"[walking, in, la, -, missing, persons]",recommendation,0.60,True,dash,False,6,walking in la,missing persons,1.0,58.5,6
9,1,Songs with the LA vibes?,80.0,Califonia Dreamin' - The Mamas &amp; The Papas,6,califonia dreamin' - the mamas & the papas,"[califonia, dreamin', -, the, mamas, &, the, p...",mixed,0.50,True,dash,False,8,califonia dreamin',the mamas & the papas,1.0,58.5,8


### Schema Enforcement

## Lectura de los parquet

In [2]:
print(pd.__version__)
print(pa.__version__)

3.0.3
16.1.0


In [3]:
paths = {
    "governance": "../datalake_gold/governance_current.parquet",
    "storytelling": "../datalake_gold/storytelling_current.parquet",
    "artists": "../datalake_silver/lastfm_top_artists/lastfm_top_artists_current.parquet",
    "tracks": "../datalake_silver/lastfm_top_tracks/lastfm_top_tracks_current.parquet",
    "reddit": "../datalake_silver/reddit/reddit_music_opinions_current.parquet"
}

In [29]:
for name, path in paths.items():
    df = pd.read_parquet(path)
    out = f"export_{name}.csv"
    df.to_csv(out, index=False)
    print(f"export_{name}.csv - {len(df)} filas, {len(df.columns)} columnas")

export_governance.csv - 73 filas, 6 columnas
export_storytelling.csv - 79 filas, 7 columnas
export_artists.csv - 150 filas, 6 columnas
export_tracks.csv - 151 filas, 10 columnas
export_reddit.csv - 228 filas, 19 columnas


In [20]:
df_sil_rute = '../datalake_gold/storytelling_current.parquet'
df_paq = pd.read_parquet(df_sil_rute)
df_paq.tail(60)

,metric_type,dim1,dim2,record_count,pct,avg_score,data_date,computed_at
280,top_track_lastfm,Don't Stop 'Til You Get Enough,Michael Jackson,8808275,0.51,1490426.0,2026-05-29,2026-05-30T11:56:18.294051
281,top_track_lastfm,Dracula - JENNIE remix,Tame Impala,5785355,0.33,705911.0,2026-05-29,2026-05-30T11:56:18.294051
282,top_track_lastfm,tHe CuRe,Olivia Rodrigo,3056147,0.18,331966.0,2026-05-29,2026-05-30T11:56:18.294051
283,top_track_lastfm,Janice STFU,Drake,1692538,0.10,282568.0,2026-05-29,2026-05-30T11:56:18.294051
284,top_track_lastfm,NATIONAL TREASURES,Drake,1260023,0.07,266633.0,2026-05-29,2026-05-30T11:56:18.294051
285,top_track_lastfm,Whisper My Name,Drake,1167070,0.07,276410.0,2026-05-29,2026-05-30T11:56:18.294051
286,top_track_lastfm,SS26,Charli xcx,685603,0.04,174213.0,2026-05-29,2026-05-30T11:56:18.294051
287,top_track_lastfm,SWIM,BTS,201114878,11.62,458121.0,2026-05-30,2026-05-30T11:56:18.294051
288,top_track_lastfm,Creep,Radiohead,61277532,3.54,4116166.0,2026-05-30,2026-05-30T11:56:18.294051
289,top_track_lastfm,"Good Luck, Babe!",Chappell Roan,60669563,3.50,2101309.0,2026-05-30,2026-05-30T11:56:18.294051


In [21]:
df_paq.head(25)

,metric_type,dim1,dim2,record_count,pct,avg_score,data_date,computed_at
0,sentiment_dist,negative,reddit,32,14.04,-0.5240,2026-05-30,2026-05-30T11:56:18.294051
1,sentiment_dist,positive,reddit,155,67.98,0.6207,2026-05-30,2026-05-30T11:56:18.294051
2,sentiment_dist,neutral,reddit,41,17.98,-0.0006,2026-05-30,2026-05-30T11:56:18.294051
3,sentiment_trend,2026-05-30,reddit,228,0.00,0.3483,2026-05-30,2026-05-30T11:56:18.294051
4,comment_type_dist,opinion,reddit,179,78.51,0.0201,2026-05-30,2026-05-30T11:56:18.294051
5,comment_type_dist,other,reddit,48,21.05,0.2167,2026-05-30,2026-05-30T11:56:18.294051
6,comment_type_dist,mixed,reddit,1,0.44,0.3500,2026-05-30,2026-05-30T11:56:18.294051
7,top_keyword,album,reddit,57,11.88,0.5561,2026-05-30,2026-05-30T11:56:18.294051
8,top_keyword,music,reddit,41,8.54,0.7276,2026-05-30,2026-05-30T11:56:18.294051
9,top_keyword,good,reddit,29,6.04,0.4413,2026-05-30,2026-05-30T11:56:18.294051


In [12]:
df_paq.info()

<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   source       73 non-null     str    
 1   field_name   73 non-null     str    
 2   kpi_type     73 non-null     str    
 3   value        73 non-null     float64
 4   unit         73 non-null     str    
 5   data_date    73 non-null     str    
 6   computed_at  73 non-null     str    
dtypes: float64(1), str(6)
memory usage: 9.5 KB
